# MSM projections: three navigation cadences read as one trend

This notebook summarizes three counterfactual means with one marginal structural model (MSM). Each
step shows its code, its output, and what the output tells you.
[MSM projections](../technical-reference/msm-projections.md) gives the projection, its clever
covariate, and the algorithm.

## The applied question

The network assigned three navigation cadences side by side.

| tier | assigned protocol | contacts in 30 days |
| --- | --- | --- |
| `low` | one transition-planning contact | 1 |
| `medium` | planning plus one follow-up contact | 2 |
| `high` | planning plus five follow-up contacts | 6 |

Nobody randomized cadence. Discharge risk and age influenced assignment and the outcome. Each
cadence uses the same script and access rules, which supports consistency. The rest of the
[shared study design](index.md#the-shared-study-design) is unchanged.

The program board asks how much the transition score changes per assigned contact. The estimand is
the straight line in assigned contacts closest to the three counterfactual means. Each cadence gets
an equal weight.

## What you will learn

| after this notebook you can | the step that shows it |
| --- | --- |
| say why unadjusted cadence means mislead | Step 3 |
| write the protocol for a three-cadence question | Step 4 |
| estimate one counterfactual mean per cadence | Step 6 |
| declare a working model and fit its coefficients | Step 7 |
| say why `cleverly` refuses to read a label as a dose | Step 8 |
| read a slope when the line misses the means | the failure mode, Step 9 |
| check the projection against a saturated model | Step 10 |
| read the diagnostics, and see where the sensitivity analysis stops | Steps 11 and 12 |

## Why this method

| your situation | what this method buys | what it costs |
| --- | --- | --- |
| three cadences and one board decision | one coefficient vector instead of one mean per cadence | the coefficients mean what the working model says. Read them as a projection |
| a contact count that you want to read as a trend | a slope with an influence curve and an interval | you code each label as a number yourself |
| a response that need not be straight | the slope is defined whether or not the line fits | the slope is not the effect of one more contact |

An outcome regression coefficient is an observed-data projection. The MSM coefficient is a
functional of the counterfactual means, and it has a value whether or not the line fits. The table
below defines the terms this notebook uses most.

| term | plain meaning |
| --- | --- |
| estimand | the number the question asks for, written before any model is chosen. See [estimands](../user-guide/estimands.md#marginal-structural-models) |
| counterfactual mean | the mean score if every eligible patient were assigned one cadence |
| working model | the line you declare, as a design with named terms. It summarizes the means and does not claim to be the true response |
| projection weight | how much each cadence counts when the line is fitted to the means. Here every cadence counts equally |
| nuisance | a model the estimate needs but the question does not ask about. Here, the outcome regression Q and the treatment mechanism g. See [point-treatment TMLE](../technical-reference/point-treatment-tmle.md) |
| saturated model | a working model with one free coefficient per cadence, so it reproduces the three means exactly |

## Step 1: set up

The setup imports the learners and prints the installed `cleverly` version. Every fit below passes
linear learners, a fold count, and a random seed. A rerun therefore reproduces the stored outputs.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.2


**What this output tells you.** The stored outputs in this notebook came from the version named
above. A different version can print different numbers.

## Step 2: the data

`make_multi_arm` draws a three-arm confounded law with known counterfactual means. The code renames
its columns to the program's names. It records the contact count of each cadence, prints the first
rows, and prints the known means.

In [2]:
from cleverly.datasets import make_multi_arm

frame, truth = make_multi_arm(n=3_000, seed=61)
frame = frame.rename(
    columns={
        "Y": "transition_score",
        "A": "cadence",
        "W1": "discharge_risk",
        "W2": "age",
        "W3": "baseline_support_need",
    }
)
ARMS = ("low", "medium", "high")
CONTACTS_30D = {"low": 1.0, "medium": 2.0, "high": 6.0}
contacts = np.array([CONTACTS_30D[arm] for arm in ARMS])
population = np.array([truth[f"ey[{arm}]"] for arm in ARMS])

print("rows and columns:", frame.shape)
print("cadence labels:", sorted(frame["cadence"].unique()))
print(frame.head().round(3))
print()
print("known counterfactual means of the synthetic law:")
for arm, value in zip(ARMS, population, strict=True):
    print(f"  ey[{arm}] ({CONTACTS_30D[arm]:.0f} contacts): {value:.4f}")
gain_per_contact = np.diff(population) / np.diff(contacts)
print(f"gain per contact, low to medium:  {gain_per_contact[0]:.2f}")
print(f"gain per contact, medium to high: {gain_per_contact[1]:.2f}")

rows and columns: (3000, 5)


cadence labels: ['high', 'low', 'medium']
   transition_score cadence  discharge_risk    age  baseline_support_need
0             1.266    high          -0.567 -1.057                  1.062
1            -0.691     low          -1.129 -1.779                  0.655
2             1.325  medium           0.841 -0.949                 -0.565
3             5.003  medium           2.780 -1.017                 -0.506
4             0.847  medium           1.179  0.745                  1.612

known counterfactual means of the synthetic law:
  ey[low] (1 contacts): -0.0000
  ey[medium] (2 contacts): 0.6000
  ey[high] (6 contacts): 1.4400
gain per contact, low to medium:  0.60
gain per contact, medium to high: 0.21


**What this output tells you.** Each row is one discharge, and `cadence` holds a text label. The
covariates are standardized (mean 0, SD 1), and the outcome is in synthetic units. The published
truths are keyed by the generator's labels, so the labels stay unchanged.

The true means are 0 for `low` (printed as `-0.0000` after rounding), 0.6000 for `medium`, and
1.4400 for `high`. The gain per contact falls from 0.60 between `low` and `medium` to 0.21 between
`medium` and `high`. A working model linear in assigned contacts is therefore misspecified on
purpose.

| feature of the law | what it means in this program |
| --- | --- |
| discharge risk and age drive assignment and the score | both are confounders |
| baseline support need drives only the score | it is not a confounder, and the adjustment set still includes it |
| the gain per contact is not constant | no straight line passes through the three means |

A real program has no `truth`. Every comparison against it below is a teaching device.

## Step 3: association first

A confounder is a variable that changes both the assigned cadence and the outcome. The code
compares the cadences before any adjustment. It prints the observed mean score, the means of the
two confounders, the share of patients, and the true counterfactual mean.

In [3]:
observed = frame.groupby("cadence")[["transition_score", "discharge_risk", "age"]].mean()
observed = observed.loc[list(ARMS)]
observed["share"] = frame["cadence"].value_counts(normalize=True).loc[list(ARMS)]
observed["true mean"] = population
print(observed.round(3))

         transition_score  discharge_risk    age  share  true mean
cadence                                                           
low                -0.082          -0.132 -0.128  0.272      -0.00
medium              1.324           0.532 -0.343  0.368       0.60
high                0.808          -0.387  0.474  0.360       1.44


**What this output tells you.** The observed means put `medium` above `high`, at 1.324 against
0.808. The true means reverse that order, at 0.60 against 1.44.

The reason is the assignment. The `medium` patients have a mean `discharge_risk` of 0.532, and the
`high` patients have -0.387. In this law a higher discharge risk raises the score, so the observed
means mix cadence with risk. A trend fitted to these observed means would summarize the confounding
too.

The shares are 0.272, 0.368, and 0.360. Step 9 uses them to show that the projection weight is part
of the estimand.

## Step 4: write the protocol

A `StudyProtocol` records the scientific design before any model runs. The record gets a
fingerprint, and every result fitted from it carries that fingerprint. This page starts from
`navigation_protocol()`, the protocol of the
[shared study design](index.md#the-shared-study-design). `dataclasses.replace` lists the three
cadences as three strategies, each with its version.

In [4]:
from dataclasses import replace

from cleverly.datasets import navigation_protocol

program = navigation_protocol()
protocol = replace(
    program,
    time_zero="Discharge-home order, after baseline measurement and before cadence assignment",
    treatment_strategies=(
        "Assign the low navigation cadence",
        "Assign the medium navigation cadence",
        "Assign the high navigation cadence",
    ),
    treatment_versions=(
        "One transition-planning contact within 30 days",
        "Transition planning plus one follow-up contact within 30 days",
        "Transition planning plus five follow-up contacts within 30 days",
    ),
    intercurrent_event_handling=(
        program.intercurrent_event_handling[0],
        "Analyze the assigned cadence regardless of completed contacts",
        program.intercurrent_event_handling[2],
    ),
    assumption_rationale=(
        "The recorded baseline variables cover the measured common causes of cadence and score",
        "Every cadence uses the same script and access rules, which supports consistency",
        program.assumption_rationale[2],
    ),
)
print("\n".join(protocol.summary_lines()))

causal study protocol: schema 1; 3858ff742f4081f5
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measurement and before cadence assignment
treatment strategies: ['Assign the low navigation cadence', 'Assign the medium navigation cadence', 'Assign the high navigation cadence']
treatment versions: ['One transition-planning contact within 30 days', 'Transition planning plus one follow-up contact within 30 days', 'Transition planning plus five follow-up contacts within 30 days']
outcome: Patient-reported transition score
horizon: 30 days after discharge
intercurrent-event handling: ['Use the transition score regardless of readmission', 'Analyze the assigned cadence regardless of completed contacts', 'The protocol scores death before day 30 as the worst transition score (composite strategy

**What this output tells you.** The first line gives the schema version and the fingerprint
`3858ff742f4081f5`. The other lines repeat each field. This page changes five fields of
`navigation_protocol()`, and
[point-treatment TMLE](point-treatment-tmle.ipynb#step-4-write-the-protocol) reads the others.

| protocol field | what this page writes |
| --- | --- |
| time zero | the discharge-home order, before cadence assignment |
| treatment strategies and versions | the three cadences, with their contacts in words |
| intercurrent-event handling | the assigned cadence is analyzed regardless of completed contacts |
| assumption rationale | the common causes of cadence and score, and one script for every cadence |

The protocol has no field for the contact mapping, the working model, or the projection weight. The
typed estimand in Step 7 owns all three. The versions state the contacts in words, and the code
states them as numbers in `CONTACTS_30D`.

## Step 5: design and identification

The design is an ordinary point-treatment design with a three-level treatment. `CounterfactualMean`
asks for one mean per cadence. The projection in Step 7 summarizes those means.

In [5]:
from cleverly import CausalStudy, CounterfactualMean, PointTreatment

study = CausalStudy(
    frame,
    design=PointTreatment(
        outcome="transition_score",
        treatment="cadence",
        adjustment=("discharge_risk", "age", "baseline_support_need"),
    ),
    protocol=protocol,
)
arms = study.identify(CounterfactualMean())
print(arms.summary())

counterfactual mean under each treatment, E[Y^a]
identified by explicit-adjustment: E_W[E(transition_score | cadence=a, W)] for a in ['high', 'low', 'medium']
adjustment/history: ['discharge_risk', 'age', 'baseline_support_need']
required nuisances: ['outcome_regression', 'treatment_mechanism']
assumptions:
  - consistency: Y = Y^a when A = a
  - no interference: one unit's potential outcome does not depend on other units' treatment assignments
  - no unmeasured confounding: Y^a is independent of A given W
  - positivity: P(cadence = a | W) > 0 almost surely for every supported treatment level a in ['high', 'low', 'medium']
causal study protocol: schema 1; 3858ff742f4081f5
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measurement and before cadence assignment
treatment strategies: ['

**What this output tells you.** The formula averages the outcome regression over all patients, once
for each cadence. The required nuisances are the outcome regression and the treatment mechanism.

| assumption | what it means for this program | can the data check it? |
| --- | --- | --- |
| consistency | each cadence always means its declared script and contacts | no |
| no interference | one patient's cadence does not change another patient's outcome | no |
| no unmeasured confounding | discharge risk, age, and support need block every common cause of cadence and score | no |
| positivity | each covariate profile has some chance of each cadence | partly, through the support report |

The positivity condition names `high`, `low`, and `medium`. Each level needs a positive probability
at every covariate value where the target places mass. The summary then repeats the stored
protocol.

## Step 6: estimate each cadence's mean

Start with the per-arm report, because the projection is a summary of it. The linear outcome model
and the multinomial logistic treatment model are correctly specified for this law.

| setting | value | what it does |
| --- | --- | --- |
| `outcome_learner` | `LinearRegression()` | fits Q, the expected score given the cadence and the covariates |
| `treatment_learner` | `LogisticRegression(max_iter=1000)` | fits g, the probability of each cadence given the covariates |
| `CrossFitting(n_folds=3)` | three folds | predicts each row from models fit without that row |
| `Runtime(random_state=61, n_jobs=1)` | fixed seed, one process | makes the fit reproducible |

Cross-fitting means each row's nuisance prediction comes from models fit without that row.
Targeting then makes a small update to Q, weighted by g, that removes first-order bias.
[CV-TMLE](../technical-reference/cv-tmle.md) and
[targeting and bounds](../user-guide/methods-learners.md#targeting-and-bounds) define both.

In [6]:
from cleverly import CrossFitting, ModelSpec, Runtime, TMLEMethod

method = TMLEMethod(
    models=ModelSpec(
        outcome_learner=LinearRegression(),
        treatment_learner=LogisticRegression(max_iter=1000),
    ),
    cross_fitting=CrossFitting(n_folds=3),
    runtime=Runtime(random_state=61, n_jobs=1),
)
arm_result = arms.estimate(method=method)

arm_table = arm_result.to_frame().set_index("estimand").loc[[f"ey[{arm}]" for arm in ARMS]]
arm_table = arm_table[["psi", "std_err", "ci_lower", "ci_upper"]]
arm_table["true mean"] = population
print(arm_table.round(4))
print()
print("protocol fingerprint on the fit:", arm_result.provenance.protocol_fingerprint)

               psi  std_err  ci_lower  ci_upper  true mean
estimand                                                  
ey[low]    -0.0069   0.0373   -0.0800    0.0662      -0.00
ey[medium]  0.6398   0.0389    0.5634    0.7161       0.60
ey[high]    1.4614   0.0404    1.3823    1.5406       1.44

protocol fingerprint on the fit: 3858ff742f4081f5


**What this output tells you.** Each row is one counterfactual mean. The `medium` estimate is 0.6398
with the interval (0.5634, 0.7161), and the true mean is 0.60. The `high` estimate is 1.4614, and
the `low` estimate is -0.0069.

Each interval contains its true mean on this draw. That is one draw, not a coverage result. The
interval comes from the influence curve, which measures how much each row moves the estimate. See
[inference](../technical-reference/inference.md). The fit carries the protocol fingerprint
`3858ff742f4081f5`.

## Step 7: declare the working model and fit the trend

The page writes the working model out. The `design` function receives one cadence label and the
covariate frame. It returns the design matrix for that cadence, with one column per term. The
`terms` tuple names the columns.

This declaration leaves `weights` unset, which sets the projection weight h(a, V) to 1 for every
cadence. The same `method` fits the projection.

In [7]:
from cleverly import MSMProjection
from cleverly.msm import MSM


def contacts_design(arm, data):
    """One row per patient: an intercept and the cadence's assigned contact count."""
    return np.column_stack([np.ones(len(data)), np.full(len(data), CONTACTS_30D[arm])])


trend = MSM(design=contacts_design, terms=("(intercept)", "assigned contacts"))
trend_effect = study.identify(MSMProjection(trend))
print("estimand:", trend_effect.summary().splitlines()[0])
print("assumptions:")
for assumption in trend_effect.identification.assumptions:
    print("  -", assumption)
print()

trend_result = trend_effect.estimate(method=method)
print(trend_result.summary())

estimand: projection of counterfactual means onto a working model
assumptions:
  - consistency: Y = Y^a when A = a
  - no interference: one unit's potential outcome does not depend on other units' treatment assignments
  - no unmeasured confounding: Y^a is independent of A given W
  - positivity: P(cadence = a | W) > 0 almost surely for every supported treatment level a in ['high', 'low', 'medium']
  - the working model and its weights are known functions of (a, V): neither phi nor h depends on the observed-data law, so the influence function carries no term for estimating them
  - the weighted Gram matrix is invertible, so the projection is a single coefficient vector rather than a set of them

Targeted maximum likelihood estimation
n = 3000; covariates = 3; arm shares: high=0.36, low=0.272, medium=0.368
causal estimand: projection of counterfactual means onto a working model
identification: explicit-adjustment; identified msm-indexed plug-in functional of E(transition_score | cadence

**What this output tells you.** The estimand is a projection of counterfactual means onto a working
model. The identification repeats the four causal assumptions and adds two.

| added assumption | what it means here |
| --- | --- |
| the working model and its weights are known functions | `contacts_design` and the uniform weight do not depend on the data |
| the weighted Gram matrix is invertible | the three cadences have distinct contact counts, so one line is closest |

The result table reports two coefficients, `msm[(intercept)]` and `msm[assigned contacts]`. The
slope is 0.26865 transition-score units per additional assigned contact. The intercept is -0.1088,
the line's value at zero contacts, which no cadence assigns.

With a uniform weight, the slope is a fixed contrast of the three means. The contrast is
(-2 E[Y(low)] - E[Y(medium)] + 3 E[Y(high)]) / 14. Step 9 checks that identity on the population means. An
inverse-probability-weighted regression with stabilized weights uses h(a) = P(A = a) instead. Its
slope is a different estimand when the cadence shares are unequal and the line does not fit.

## Step 8: the contact mapping is the program's decision

The contact counts are the program's decision, not the estimator's. The `MSM.linear` constructor builds a model
linear in the arm, so it reads each arm label as a number. The code asks for it on the text labels.

In [8]:
from cleverly.exceptions import DataError

try:
    study.identify(MSMProjection(MSM.linear())).estimate(method=method)
except DataError as error:
    refusal = str(error)
    print("refused:", refusal)
else:
    raise AssertionError("MSM.linear accepted text cadence labels")

refused: MSM.linear reads the treatment level 'high' as a number, and it is not one. A working model linear in the arm treats it as a dose to interpolate between, which a label has no ordering for -- and the sort order a coding would fall back on is not one anybody chose. Pass design= and code the arms explicitly.


**What this output tells you.** `cleverly` refuses with a `DataError`. The message says that `'high'`
is not a number, and it asks for an explicit `design=`.

The refusal does not fall back on sort order, because nobody chose the order of
`{"high", "low", "medium"}`. A `0, 1, 2` coding would also be wrong here, because the real spacing
of 1, 2, and 6 contacts is not even. The
[scope and refusals](../technical-reference/scope-and-refusals.md#how-to-read-a-refusal) page
explains how to read a refusal.

## Step 9: the failure mode, a projection is not a fitted curve

The working model from Step 7 is wrong for this law.

The code compares the line with the means, first in the population and then in the fits. It also
computes the population slope as the fixed contrast of the three means. It then weights each
cadence by its share.

In [9]:
design = np.column_stack([np.ones(3), contacts])
projection, *_ = np.linalg.lstsq(design, population, rcond=None)
names = ("msm[(intercept)]", "msm[assigned contacts]")
estimated_line = design @ np.array([trend_result[name].psi for name in names])
arm_means = np.array([arm_result[f"ey[{arm}]"].psi for arm in ARMS])

in_population = pd.DataFrame(
    {
        "contacts": contacts,
        "population mean": population,
        "population line": design @ projection,
        "line minus mean": design @ projection - population,
    },
    index=list(ARMS),
)
in_fits = pd.DataFrame(
    {
        "estimated mean": arm_means,
        "ci_lower": [arm_result[f"ey[{arm}]"].ci[0] for arm in ARMS],
        "ci_upper": [arm_result[f"ey[{arm}]"].ci[1] for arm in ARMS],
        "estimated line": estimated_line,
        "line minus mean": estimated_line - arm_means,
    },
    index=list(ARMS),
)
print("in the population:")
print(in_population.round(4).to_string())
print()
print("in the fits:")
print(in_fits.round(4).to_string())
medium = arm_result["ey[medium]"]
gap_in_se = (medium.ci[0] - estimated_line[1]) / medium.std_error
print(f"medium: the line sits {gap_in_se:.1f} standard errors below the interval's lower limit")
print()
print(f"population projection: intercept {projection[0]:.4f}, slope {projection[1]:.4f}")
contrast_slope = (-2 * population[0] - population[1] + 3 * population[2]) / 14
print(f"slope as the fixed contrast of the three means: {contrast_slope:.4f}")

shares = observed["share"].to_numpy()
root = np.sqrt(shares)
share_projection, *_ = np.linalg.lstsq(design * root[:, None], population * root, rcond=None)
print(f"population slope if each cadence were weighted by its share: {share_projection[1]:.4f}")
print()
for name, target in zip(names, projection, strict=True):
    low, high = trend_result[name].ci
    print(f"{name}: CI=({low:.4f}, {high:.4f})  population projection {target:.4f}")

in the population:
        contacts  population mean  population line  line minus mean
low          1.0            -0.00           0.1486           0.1486
medium       2.0             0.60           0.4143          -0.1857
high         6.0             1.44           1.4771           0.0371

in the fits:
        estimated mean  ci_lower  ci_upper  estimated line  line minus mean
low            -0.0069   -0.0800    0.0662          0.1598           0.1667
medium          0.6398    0.5634    0.7161          0.4285          -0.2113
high            1.4614    1.3823    1.5406          1.5031           0.0417
medium: the line sits 3.5 standard errors below the interval's lower limit

population projection: intercept -0.1171, slope 0.2657
slope as the fixed contrast of the three means: 0.2657
population slope if each cadence were weighted by its share: 0.2593

msm[(intercept)]: CI=(-0.1828, -0.0348)  population projection -0.1171
msm[assigned contacts]: CI=(0.2510, 0.2863)  population projectio

**What this output tells you.** In the population, the line misses `medium` by about 0.19, since
`line minus mean` is -0.1857. It overshoots `low` by 0.1486 and `high` by 0.0371. The line
minimizes the total squared deviation from the three means under the uniform weight.

In the fits, the estimated line misses the estimated `medium` mean by 0.2113. It sits 3.5 standard
errors below the lower limit of the `medium` interval. The fitted line does not describe the
`medium` cadence.

| check | value on this draw |
| --- | --- |
| population projection | intercept -0.1171, slope 0.2657 |
| slope as the fixed contrast of the means | 0.2657 |
| slope interval from the fit | (0.2510, 0.2863), which contains 0.2657 |
| intercept interval from the fit | (-0.1828, -0.0348), which contains -0.1171 |
| population slope under the share weight | 0.2593 |

The estimand is the projection, and the estimator recovers it on this draw. Read the slope as "the
best linear summary of the cadence response under a uniform weight". Do not read it as "the causal
effect of one more contact".

| consequence | why |
| --- | --- |
| the coefficient depends on the working model | change the terms and you change the estimand, not just the estimate |
| the coefficient depends on the weight | the projection minimizes a weighted squared error. The share weight gives 0.2593 instead of 0.2657 |
| misspecification is not a bug in the fit | the parameter is well defined either way. Interval validity still needs the causal and nuisance conditions |

The share weight in this step uses the true means and a known number, so it is only an illustration.
A weight estimated from the treatment mechanism is refused, as the
[technical entry](../technical-reference/msm-projections.md#variations) explains.

A board that reads the slope as "each extra contact buys this much" will extrapolate to ten
contacts. No patient in the study was assigned ten contacts, and the projection says nothing about
that cadence.

## Step 10: the control, a saturated working model

A saturated model has one free coefficient per cadence. It cannot be misspecified, so it represents
the three means exactly.

The code fits it beside the `ATE` contrasts against `low`, with the same method. Each row pairs one
coefficient with its counterpart.

In [10]:
from cleverly import ATE


def cadence_indicators(arm, data):
    """One free coefficient per cadence: an intercept and two cadence indicators."""
    n = len(data)
    return np.column_stack(
        [
            np.ones(n),
            np.full(n, float(arm == "medium")),
            np.full(n, float(arm == "high")),
        ]
    )


saturated = MSM(design=cadence_indicators, terms=("(intercept)", "medium vs low", "high vs low"))
saturated_result = study.identify(MSMProjection(saturated)).estimate(method=method)
arm_contrasts = study.identify(ATE(reference="low")).estimate(method=method)

pairs = (
    ("msm[(intercept)]", arm_result, "ey[low]"),
    ("msm[medium vs low]", arm_contrasts, "ate[medium vs low]"),
    ("msm[high vs low]", arm_contrasts, "ate[high vs low]"),
)
rows = []
for name, counterpart, key in pairs:
    coefficient, match = saturated_result[name], counterpart[key]
    rows.append(
        {
            "coefficient": name,
            "psi": coefficient.psi,
            "ci": f"({coefficient.ci[0]:.4f}, {coefficient.ci[1]:.4f})",
            "counterpart": key,
            "counterpart psi": match.psi,
            "counterpart ci": f"({match.ci[0]:.4f}, {match.ci[1]:.4f})",
            "influence curves agree within 1e-12": bool(
                np.allclose(coefficient.influence_curve, match.influence_curve, rtol=0, atol=1e-12)
            ),
        }
    )
print(pd.DataFrame(rows).round(4).to_string(index=False))

       coefficient     psi                ci        counterpart  counterpart psi    counterpart ci  influence curves agree within 1e-12
  msm[(intercept)] -0.0069 (-0.0800, 0.0662)            ey[low]          -0.0069 (-0.0800, 0.0662)                                 True
msm[medium vs low]  0.6466  (0.5573, 0.7360) ate[medium vs low]           0.6466  (0.5573, 0.7360)                                 True
  msm[high vs low]  1.4683  (1.3769, 1.5597)   ate[high vs low]           1.4683  (1.3769, 1.5597)                                 True


**What this output tells you.** The intercept is -0.0069, the same value as the `low` mean in
Step 6. The other coefficients are 0.6466 and 1.4683, the same as the `medium` versus `low` and
`high` versus `low` contrasts. The intervals match. Each pair of influence curves agrees within `1e-12` at every row, so the
standard errors match too.

This check shows that the projection is a reparameterization, not a different analysis. It is also
the practical fallback. A board that does not want to commit to a spacing can report the saturated
model.

A saturated check cannot test a non-saturated projection. A saturated model fits, so it agrees with
the arm means whatever the projection code does. The
[technical entry](../technical-reference/msm-projections.md#validation-issues-special-to-this-method)
describes the checks that do test it.

## Step 11: diagnostics, what the fit can check

`include_retargets=True` runs the truncation curve with the assessment. Truncation clips each
fitted cadence probability into a bound, so no row gets an extreme weight. The curve retargets each
coefficient at a range of bounds without refitting the nuisance models.

Positivity means that every kind of patient has some chance of each cadence. The support report
describes positivity in the fitted data. See
[diagnostics](../user-guide/results-assessment.md#diagnostics).

In [11]:
assessment = trend_result.assess(include_retargets=True)
print(assessment.summary())
print("needs attention:", tuple(item.name for item in assessment.attention))

support = assessment.report("support")
scores = assessment.report("score_equations")
curve = assessment.report("truncation_curve")
print()
print(support.summary())
print()
print(scores.summary())
print()
slope_curve = curve.loc[
    curve["estimand"] == "msm[assigned contacts]",
    ["bound", "psi", "delta_from_fitted", "ci_lower", "ci_upper", "truncated_fraction"],
]
print(slope_curve.round(4).to_string(index=False))

Returned results
----------------
surface      operation         result                                                                                                                                                                                 
-----------  ----------------  ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
validation   nuisance_models   4 nuisance model report(s) are available                                                                                                                                               
diagnostics  truncation_curve  2 parameter(s) over evaluated lower bounds [0.001, 0.2]; signed movement from the fitted estimate: msm[(intercept)] [-0.002739, 0.001783], msm[assigned contacts] [-0.002927, 0.001645]

Checks
------
status   count  operations                
-------  -----  --------------------------
warni

**What this output tells you.** Read the parts in order.

| output part | what it shows on this draw |
| --- | --- |
| `Checks` and `needs attention` | one warning, `support`. The score-equation check passed |
| `Not run` | the omitted-variable operations are `unavailable` for this fit. Step 12 shows why |
| positivity and overlap | 37 units (1.23%) sit at the truncation bound. The narrowest arm, `high`, has an ESS / n of 0.448 |
| the verdict | "some truncation". The report reads it from the share of units the fit truncated |
| score-equation check | one fluctuation equation and one influence-curve row per coefficient, not one per arm |
| truncation curve | the slope stays between 0.2657 and 0.2703 across bounds from 0.0010 to 0.2000 |

The largest bound clips a fraction of 0.6133 of the rows, and the slope still moves by only 0.0029.
Limited movement shows stability under this regularization choice. It does not verify positivity,
and it says nothing about a cadence outside the observed range.

## Step 12: sensitivity, and where it stops

Sensitivity analysis asks how strong an unmeasured confounder would need to be to change the
conclusion. [Sensitivity analysis](../user-guide/results-assessment.md#sensitivity-analysis) defines
it. The code asks for the robustness value of the trend fit, then of each arm contrast from Step 10.

In [12]:
from cleverly.exceptions import CapabilityError

try:
    trend_result.sensitivity.robustness_value()
except CapabilityError as error:
    sensitivity_refusal = str(error)
    print("refused for the slope:", sensitivity_refusal)
else:
    raise AssertionError("the omitted-variable bound accepted an MSM coefficient")
print()

robustness = {
    name: arm_contrasts.sensitivity.robustness_value(estimand=name)
    for name in ("ate[medium vs low]", "ate[high vs low]")
}
for name, values in robustness.items():
    print(
        f"{name}: robustness value {values['rv']:.3f} (confidence-limit value {values['rva']:.3f})"
    )

refused for the slope: the omitted-variable bound applies to the arm-indexed linear estimands, and this fit reports none: its parameters are indexed by 'msm' and it reported ['msm[(intercept)]', 'msm[assigned contacts]']. The bias is bounded through the Riesz representer of a mean or a contrast of arms, which a fit whose counterfactuals are not arms does not have.

ate[medium vs low]: robustness value 0.205 (confidence-limit value 0.182)
ate[high vs low]: robustness value 0.426 (confidence-limit value 0.395)


**What this output tells you.** `cleverly` refuses the bound for the trend fit with a
`CapabilityError`. The message says that the omitted-variable bound applies to arm-indexed means
and contrasts. This fit reports MSM coefficients instead, so `cleverly` offers no bound on the
slope.

The arm contrasts do have a bound. Each robustness value assumes worst-case alignment (`rho=1`). It
is the equal outcome-side and treatment-side strength that moves the point estimate to zero.

| contrast | robustness value | confidence-limit value |
| --- | --- | --- |
| `ate[medium vs low]` | 0.205 | 0.182 |
| `ate[high vs low]` | 0.426 | 0.395 |

These values describe the saturated summary, not the slope. The `medium` versus `low` contrast
reaches zero at the weaker confounding strength. The
[omitted-variable bounds](../technical-reference/validation-methods.md#omitted-variable-bounds-robustness-value-benchmark-and-contours)
section defines each quantity.

## How far to trust this

The [point-treatment MSM projection study](../technical-reference/method-evidence/point-treatment-msm-projection.md)
validates the identity-link projection with fixed nonuniform weights. Its limits name ordinary,
non-cross-fitted targeting, three terms, and two treatment arms. This page uses three-fold
cross-fitting, three arms, and a uniform weight, so no registered study covers this exact
construction.

| layer | establishes | does not establish |
| --- | --- | --- |
| the saturated control | the projection reproduces the arm report when it can | that a non-saturated working model is a good summary |
| the score-equation report | one solved score per coefficient | that the working model resembles the truth |
| the support report | how the fitted mechanism spreads over the three cadences, and how many units it truncated | that the true mechanism is bounded away from zero, or that the coefficient answers the board's question |
| the truncation curve | whether the slope is stable across declared mechanism bounds | support for a cadence outside the observed range |
| the arm-contrast robustness values | how strong a confounder would need to be to remove each contrast | any bound on the slope itself |

These checks do not establish exchangeability or a well-chosen working model. The
[technical entry](../technical-reference/msm-projections.md#validation-issues-special-to-this-method)
links the registered evidence and its limits.

## Where to go next

The same projection works over regimens and horizons in a longitudinal fit. Its design callable also
receives the horizon, and `MSM.linear` is refused there too. Read
[longitudinal TMLE](longitudinal-tmle.ipynb) and [time-to-event outcomes](longitudinal-survival.ipynb)
first. The projection summarizes the parameters those pages estimate one at a time.

The [examples index](index.md#the-program) lists every tutorial in the program.